In [1]:
# ============================================================
# BOOTSTRAP CI — CELL 1: DIAGNOSTIC
# Find what per-customer explanation scores were saved.
# Run this first; paste the output back before running Cell 2.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# ---- adjust this if your project folder differs ----
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

SEARCH_ROOTS = [
    '/content/drive/MyDrive',
    '/content',
]

# file types that could hold saved metric arrays
PATTERNS = ['*.npy', '*.npz', '*.csv', '*.pkl', '*.joblib', '*.json']

# words that suggest explanation-evaluation output
KEYWORDS = [
    'fidelity', 'stability', 'sparsity', 'agreement', 'deletion',
    'aopc', 'shap', 'lime', 'eval', 'metric', 'per_customer',
    'percustomer', 'instance', 'r2', 'surrogate'
]

print("=" * 70)
print("SEARCHING FOR SAVED EVALUATION ARTEFACTS")
print("=" * 70)

hits = []
for root in SEARCH_ROOTS:
    if not os.path.isdir(root):
        continue
    for pattern in PATTERNS:
        for path in glob.glob(os.path.join(root, '**', pattern), recursive=True):
            name = os.path.basename(path).lower()
            if any(k in name for k in KEYWORDS):
                try:
                    size_kb = os.path.getsize(path) / 1024
                except OSError:
                    size_kb = -1
                hits.append((path, size_kb))

if not hits:
    print("\nNo obvious matches found by filename.")
else:
    print(f"\nFound {len(hits)} candidate file(s):\n")
    for path, size_kb in sorted(hits, key=lambda x: -x[1]):
        print(f"  {size_kb:9.1f} KB   {path}")

# ------------------------------------------------------------
# Inspect each candidate: we want arrays of length ~500
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("INSPECTING CANDIDATES FOR 500-LENGTH ARRAYS")
print("=" * 70)

TARGET_LEN = 500

def report_array(label, arr):
    arr = np.asarray(arr)
    flag = "  <<< LENGTH ~500" if TARGET_LEN * 0.8 <= arr.shape[0] <= TARGET_LEN * 1.2 else ""
    print(f"    {label}: shape={arr.shape}, dtype={arr.dtype}{flag}")
    if arr.ndim == 1 and arr.size:
        try:
            print(f"       first 5: {np.round(arr[:5].astype(float), 4)}")
            print(f"       mean={float(np.nanmean(arr)):.4f}")
        except (ValueError, TypeError):
            print(f"       first 5: {arr[:5]}")

for path, _ in hits:
    print(f"\n--- {path}")
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == '.npy':
            report_array('array', np.load(path, allow_pickle=True))

        elif ext == '.npz':
            with np.load(path, allow_pickle=True) as z:
                print(f"    keys: {list(z.keys())}")
                for k in z.keys():
                    report_array(k, z[k])

        elif ext == '.csv':
            df = pd.read_csv(path)
            print(f"    shape={df.shape}")
            print(f"    columns: {list(df.columns)}")
            if TARGET_LEN * 0.8 <= len(df) <= TARGET_LEN * 1.2:
                print("    <<< ROW COUNT ~500")
                print(df.head(3).to_string())

        elif ext in ('.pkl', '.joblib'):
            import joblib
            obj = joblib.load(path)
            print(f"    type: {type(obj)}")
            if isinstance(obj, dict):
                print(f"    keys: {list(obj.keys())[:25]}")
                for k, v in list(obj.items())[:25]:
                    if isinstance(v, (list, np.ndarray)):
                        report_array(k, v)
                    else:
                        print(f"    {k}: {type(v).__name__} = {str(v)[:60]}")
            elif isinstance(obj, pd.DataFrame):
                print(f"    DataFrame shape={obj.shape}, columns={list(obj.columns)}")
            elif isinstance(obj, (list, np.ndarray)):
                report_array('object', obj)

        elif ext == '.json':
            import json
            with open(path) as f:
                obj = json.load(f)
            if isinstance(obj, dict):
                print(f"    keys: {list(obj.keys())[:25]}")
                for k, v in obj.items():
                    if isinstance(v, list):
                        report_array(k, v)
                    else:
                        print(f"    {k}: {v}")
    except Exception as e:
        print(f"    [could not read: {type(e).__name__}: {e}]")

print("\n" + "=" * 70)
print("DONE")
print("Looking for: arrays/columns of length ~500 holding per-customer")
print("fidelity, stability, or LIME R-squared values.")
print("=" * 70)

Mounted at /content/drive
SEARCHING FOR SAVED EVALUATION ARTEFACTS

Found 80 candidate file(s):

    17528.1 KB   /content/drive/MyDrive/MSc_Dissertation/models/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/outputs/results/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/marker_test/code/outputs/results/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/project_backup_15aug/Documents/MSc_Dissertation/code/outputs/results/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/MSc_Dissertation/models/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/outputs/results/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/marker_test/code/outputs/results/shap_outputs.pkl
    17528.1 KB   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation

In [4]:
# ============================================================
# BOOTSTRAP CI — CELL 2
# Bootstraps sparsity and agreement from saved per-customer
# attributions. No model reload, no re-running SHAP/LIME.
# ============================================================

import numpy as np
import pandas as pd
import os
import joblib
from scipy.stats import spearmanr

RESULTS = '/content/drive/MyDrive/MSc_Dissertation/models'
RNG_SEED = 42
N_BOOT = 2000
rng = np.random.default_rng(RNG_SEED)

# ------------------------------------------------------------
# PART A — print exactly what was originally reported
# ------------------------------------------------------------
print("=" * 70)
print("A. ORIGINAL REPORTED METRICS (full contents)")
print("=" * 70)

em = joblib.load(os.path.join(RESULTS, 'evaluation_metrics.pkl'))
for section in ['fidelity', 'stability', 'sparsity', 'agreement']:
    print(f"\n[{section}]")
    val = em.get(section)
    if isinstance(val, dict):
        for k, v in val.items():
            print(f"   {k}: {v}")
    else:
        print(f"   {val}")
print(f"\n[lime_local_r2_xgb]: {em.get('lime_local_r2_xgb')}")

# ------------------------------------------------------------
# PART B — load aligned attribution matrices
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("B. LOADING ATTRIBUTIONS")
print("=" * 70)

shap_df = pd.read_csv(os.path.join(RESULTS, 'xgb_shap_eval.csv'), index_col=0)
lime_df = pd.read_csv(os.path.join(RESULTS, 'lime_eval.csv'), index_col=0)

assert list(shap_df.columns) == list(lime_df.columns), "Feature columns differ"
assert (shap_df.index == lime_df.index).all(), "Customer IDs not aligned"

S = shap_df.values.astype(float)   # (500, 66) SHAP attributions
L = lime_df.values.astype(float)   # (500, 66) LIME attributions
n, p = S.shape
features = list(shap_df.columns)

print(f"   Aligned: {n} customers x {p} features")
print(f"   First 3 customer IDs: {list(shap_df.index[:3])}")

# ------------------------------------------------------------
# PART C — calibrate the "active feature" threshold
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("C. CALIBRATING SPARSITY THRESHOLD")
print("=" * 70)

sp = em.get('sparsity', {})
target_shap = None
target_lime = None
for k, v in sp.items():
    kl = k.lower()
    if 'shap' in kl:
        target_shap = float(v)
    elif 'lime' in kl:
        target_lime = float(v)
print(f"   Target means -> SHAP: {target_shap}, LIME: {target_lime}")

def active_counts_abs(M, tol):
    return (np.abs(M) > tol).sum(axis=1).astype(float)

def active_counts_rel(M, frac):
    mx = np.abs(M).max(axis=1, keepdims=True)
    mx[mx == 0] = 1.0
    return (np.abs(M) > frac * mx).sum(axis=1).astype(float)

candidates = []
for tol in [0.0, 1e-12, 1e-8, 1e-6, 1e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]:
    cs, cl = active_counts_abs(S, tol), active_counts_abs(L, tol)
    err = abs(cs.mean() - (target_shap or 0)) + abs(cl.mean() - (target_lime or 0))
    candidates.append(('abs', tol, cs.mean(), cl.mean(), err))
for frac in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]:
    cs, cl = active_counts_rel(S, frac), active_counts_rel(L, frac)
    err = abs(cs.mean() - (target_shap or 0)) + abs(cl.mean() - (target_lime or 0))
    candidates.append(('rel', frac, cs.mean(), cl.mean(), err))

candidates.sort(key=lambda x: x[4])
print(f"\n   {'kind':5s} {'thresh':>10s} {'SHAP mean':>10s} {'LIME mean':>10s} {'error':>8s}")
for kind, t, ms, ml, err in candidates[:8]:
    print(f"   {kind:5s} {t:10.2e} {ms:10.3f} {ml:10.3f} {err:8.3f}")

best_kind, best_t, bs, bl, best_err = candidates[0]
print(f"\n   BEST: {best_kind} threshold {best_t:.2e}  ->  SHAP {bs:.3f}, LIME {bl:.3f}")
if best_err > 1.0:
    print("   WARNING: does not closely reproduce reported means. Report this back.")

if best_kind == 'abs':
    shap_active = active_counts_abs(S, best_t)
    lime_active = active_counts_abs(L, best_t)
else:
    shap_active = active_counts_rel(S, best_t)
    lime_active = active_counts_rel(L, best_t)

# ------------------------------------------------------------
# PART D — per-customer agreement
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("D. PER-CUSTOMER AGREEMENT")
print("=" * 70)

def topk_overlap(a, b, k):
    ta = set(np.argsort(-np.abs(a))[:k])
    tb = set(np.argsort(-np.abs(b))[:k])
    return len(ta & tb) / k

overlap5 = np.array([topk_overlap(S[i], L[i], 5) for i in range(n)])
overlap10 = np.array([topk_overlap(S[i], L[i], 10) for i in range(n)])

rank_corr = np.empty(n)
for i in range(n):
    rc = spearmanr(S[i], L[i]).correlation
    rank_corr[i] = 0.0 if np.isnan(rc) else rc

print(f"   Feature overlap@5   mean = {overlap5.mean():.4f}")
print(f"   Feature overlap@10  mean = {overlap10.mean():.4f}")
print(f"   Signed rank corr    mean = {rank_corr.mean():.4f}")

# ------------------------------------------------------------
# PART E — bootstrap
# ------------------------------------------------------------
print("\n" + "=" * 70)
print(f"E. BOOTSTRAP ({N_BOOT} resamples, seed {RNG_SEED})")
print("=" * 70)

boot_idx = rng.integers(0, n, size=(N_BOOT, n))

def boot_mean(x):
    d = np.asarray(x)[boot_idx].mean(axis=1)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def boot_paired_diff(x, y):
    """Bootstrap the paired difference x - y."""
    d = (np.asarray(x) - np.asarray(y))[boot_idx].mean(axis=1)
    lo, hi = np.percentile(d, 2.5), np.percentile(d, 97.5)
    p_two = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), lo, hi, max(p_two, 1.0 / N_BOOT)

rows = []

m, lo, hi = boot_mean(shap_active)
rows.append(('Sparsity - SHAP active features', m, lo, hi, ''))
m, lo, hi = boot_mean(lime_active)
rows.append(('Sparsity - LIME active features', m, lo, hi, ''))
d, lo, hi, pv = boot_paired_diff(shap_active, lime_active)
rows.append(('Sparsity - SHAP minus LIME', d, lo, hi, f'p={pv:.4f}'))

m, lo, hi = boot_mean(overlap5)
rows.append(('Agreement - feature overlap@5', m, lo, hi, ''))
m, lo, hi = boot_mean(overlap10)
rows.append(('Agreement - feature overlap@10', m, lo, hi, ''))
m, lo, hi = boot_mean(rank_corr)
rows.append(('Agreement - signed rank corr', m, lo, hi, ''))

print(f"\n   {'Quantity':38s} {'Mean':>9s} {'95% CI low':>11s} {'95% CI high':>12s}  {'sig':<12s}")
print("   " + "-" * 86)
for label, m, lo, hi, extra in rows:
    print(f"   {label:38s} {m:9.4f} {lo:11.4f} {hi:12.4f}  {extra:<12s}")

# ------------------------------------------------------------
# PART F — interpretation
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("F. READS")
print("=" * 70)

d, lo, hi, pv = boot_paired_diff(shap_active, lime_active)
if hi < 0:
    print(f"   SPARSITY: SHAP significantly sparser than LIME "
          f"(diff {d:.2f}, 95% CI [{lo:.2f}, {hi:.2f}], p={pv:.4f}).")
elif lo > 0:
    print(f"   SPARSITY: LIME significantly sparser (unexpected).")
else:
    print(f"   SPARSITY: difference NOT significant (CI spans zero).")

_, rlo, rhi = boot_mean(rank_corr)
if rlo < 0 < rhi:
    print(f"   AGREEMENT: rank correlation CI [{rlo:.3f}, {rhi:.3f}] includes zero "
          f"-> statistically indistinguishable from no association. Strong result.")
else:
    print(f"   AGREEMENT: rank correlation CI [{rlo:.3f}, {rhi:.3f}] excludes zero.")

_, olo, ohi = boot_mean(overlap10)
print(f"   AGREEMENT: overlap@10 = {overlap10.mean():.3f}, 95% CI [{olo:.3f}, {ohi:.3f}].")

# ------------------------------------------------------------
# PART G — save
# ------------------------------------------------------------
per_cust = pd.DataFrame({
    'customer_id': shap_df.index,
    'shap_active': shap_active,
    'lime_active': lime_active,
    'overlap_at_5': overlap5,
    'overlap_at_10': overlap10,
    'rank_corr': rank_corr,
})
out_csv = os.path.join(RESULTS, 'per_customer_metrics.csv')
per_cust.to_csv(out_csv, index=False)

summary = pd.DataFrame(rows, columns=['quantity', 'mean', 'ci_low', 'ci_high', 'note'])
out_sum = os.path.join(RESULTS, 'bootstrap_ci_summary.csv')
summary.to_csv(out_sum, index=False)

joblib.dump({
    'n_boot': N_BOOT, 'seed': RNG_SEED, 'n_customers': n,
    'sparsity_threshold_kind': best_kind, 'sparsity_threshold': best_t,
    'per_customer': per_cust, 'summary': summary,
}, os.path.join(RESULTS, 'bootstrap_results.pkl'))

print("\n" + "=" * 70)
print(f"Saved: {out_csv}")
print(f"Saved: {out_sum}")
print("DONE — paste sections A, C, E and F back.")
print("=" * 70)

A. ORIGINAL REPORTED METRICS (full contents)

[fidelity]
   SHAP deletion-AOPC: 0.5814210176467896
   LIME deletion-AOPC: 0.5377316474914551
   Random deletion-AOPC: 0.38310492038726807

[stability]
   SHAP input-stability (cosine): 0.4335164671432296
   LIME input-stability (cosine): 0.3230816807893799
   LIME seed-stability (cosine): 0.46627826702944786

[sparsity]
   SHAP mean active features: 28.632
   LIME mean active features: 48.226
   SHAP median: 29.0
   LIME median: 51.0

[agreement]
   Feature agreement@5: 0.1372
   Rank agreement@5: 0.021200000000000004
   Sign agreement@5: 0.13099041533546327
   Feature agreement@10: 0.149
   Rank agreement@10: 0.0144
   Sign agreement@10: 0.18765560165975104
   Signed rank corr (mean): -0.061308088925999385

[lime_local_r2_xgb]: 0.125

B. LOADING ATTRIBUTIONS
   Aligned: 500 customers x 66 features
   First 3 customer IDs: [49649, 48978, 22409]

C. CALIBRATING SPARSITY THRESHOLD
   Target means -> SHAP: 29.0, LIME: 51.0

   kind      thre

In [5]:
# ============================================================
# BOOTSTRAP CI - CELL 3: LOCATE ORIGINAL METRIC CODE
# Reads the notebooks to recover the exact deletion-AOPC and
# stability definitions, so recomputation matches Chapter 5.
# ============================================================

import os
import glob
import json

BAR = "=" * 70
SEARCH_ROOTS = ['/content/drive/MyDrive']
KEYWORDS = ['aopc', 'deletion', 'fidelity', 'stability', 'active_feat', 'sparsity']

print(BAR)
print("A. NOTEBOOKS FOUND")
print(BAR)

nbs = []
for root in SEARCH_ROOTS:
    for path in glob.glob(os.path.join(root, '**', '*.ipynb'), recursive=True):
        if '.ipynb_checkpoints' in path:
            continue
        nbs.append(path)

seen_names = set()
unique_nbs = []
for path in sorted(nbs):
    base = os.path.basename(path)
    if base in seen_names:
        continue
    seen_names.add(base)
    unique_nbs.append(path)

for path in unique_nbs:
    print("   " + path)

print("")
print(BAR)
print("B. CODE CELLS MENTIONING METRIC KEYWORDS")
print(BAR)

for path in unique_nbs:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            nb = json.load(f)
    except Exception as e:
        print("")
        print("   [could not read " + os.path.basename(path) + ": " + str(e) + "]")
        continue

    cells = nb.get('cells', [])
    matches = []
    for i, cell in enumerate(cells):
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source', []))
        low = src.lower()
        if any(kw in low for kw in KEYWORDS):
            matches.append((i, src))

    if not matches:
        continue

    print("")
    print("#" * 70)
    print("# NOTEBOOK: " + os.path.basename(path))
    print("# matching code cells: " + str(len(matches)))
    print("#" * 70)

    for i, src in matches:
        print("")
        print("--- cell " + str(i) + " " + "-" * 50)
        lines = src.split("\n")
        if len(lines) > 120:
            for ln in lines[:120]:
                print(ln)
            print("... [truncated, " + str(len(lines) - 120) + " more lines]")
        else:
            for ln in lines:
                print(ln)

print("")
print(BAR)
print("DONE")
print("deletion-AOPC / fidelity or the stability computation.")
print(BAR)

A. NOTEBOOKS FOUND
   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/notebooks/Day1_Hyperparameter_Tuning.ipynb
   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/notebooks/Day2_SHAP_Explanations.ipynb
   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/notebooks/Day3_LIME_Explanations.ipynb
   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/notebooks/Day4_Evaluation_Metrics.ipynb
   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/notebooks/Day5_SHAP_Segmentation.ipynb
   /content/drive/MyDrive/03189049_MSc_AICS_Dissertation/Documents/MSc_Dissertation/code/notebooks/Export_Figures.ipynb
   /content/drive/MyDrive/Colab Notebooks/Bootstrap.ipynb
   /content/drive/MyDrive/Colab Notebooks/ChurnPrediction.ipynb
   /content/drive/MyDrive/Colab Notebooks/Copy of LDSCI7234 Programming for Data Applications 

In [6]:
# ============================================================
# BOOTSTRAP CI - CELL 4: FIDELITY
# Recomputes deletion-AOPC per customer using the exact Day 4
# definition, verifies against reported values, then bootstraps.
# ============================================================

import os
import glob
import pickle
import numpy as np
import pandas as pd

RESULTS = '/content/drive/MyDrive/MSc_Dissertation/models'
FID_K = 15
RNG_SEED = 42
N_BOOT = 2000
BAR = "=" * 70

print(BAR)
print("A. FILES IN MODELS DIRECTORY")
print(BAR)
for f in sorted(os.listdir(RESULTS)):
    size_kb = os.path.getsize(os.path.join(RESULTS, f)) / 1024.0
    print("   " + format(size_kb, '9.1f') + " KB   " + f)

# ------------------------------------------------------------
# B. Load model, test data, eval set
# ------------------------------------------------------------
print("")
print(BAR)
print("B. LOADING MODEL AND TEST DATA")
print(BAR)

def try_load(path):
    try:
        with open(path, 'rb') as fh:
            return pickle.load(fh)
    except Exception as e:
        print("   [failed " + os.path.basename(path) + ": " + str(e) + "]")
        return None

# test data
test_obj = None
for cand in ['test_data.pkl']:
    p = os.path.join(RESULTS, cand)
    if os.path.exists(p):
        test_obj = try_load(p)
        print("   loaded " + cand + " -> " + str(type(test_obj)))
        break

if test_obj is None:
    for p in glob.glob(os.path.join(RESULTS, '*test*.pkl')):
        test_obj = try_load(p)
        if test_obj is not None:
            print("   loaded " + os.path.basename(p))
            break

if isinstance(test_obj, dict):
    print("   test keys: " + str(list(test_obj.keys())))

X_test_scaled = None
if isinstance(test_obj, dict):
    for k in ['X_test_scaled', 'X_test', 'Xtest', 'X']:
        if k in test_obj:
            X_test_scaled = test_obj[k]
            print("   using X from key: " + k)
            break
elif isinstance(test_obj, (tuple, list)):
    for item in test_obj:
        if isinstance(item, pd.DataFrame):
            X_test_scaled = item
            print("   using first DataFrame in tuple")
            break

if X_test_scaled is None:
    raise RuntimeError("Could not locate X_test_scaled - paste section A back.")

X_test_scaled = pd.DataFrame(X_test_scaled)
print("   X_test_scaled shape: " + str(X_test_scaled.shape))

# xgboost model
xgb_model = None
for pat in ['xgboost_tuned.pkl', 'xgb_tuned.pkl', '*xgb*tuned*.pkl', '*xgboost*.pkl']:
    for p in glob.glob(os.path.join(RESULTS, pat)):
        obj = try_load(p)
        if obj is not None and hasattr(obj, 'predict_proba'):
            xgb_model = obj
            print("   loaded model: " + os.path.basename(p))
            break
    if xgb_model is not None:
        break

if xgb_model is None:
    raise RuntimeError("Could not locate tuned XGBoost model - paste section A back.")

# attributions
shap_eval = pd.read_csv(os.path.join(RESULTS, 'xgb_shap_eval.csv'), index_col=0)
lime_eval = pd.read_csv(os.path.join(RESULTS, 'lime_eval.csv'), index_col=0)
feature_names = list(shap_eval.columns)
lime_eval = lime_eval.reindex(columns=feature_names)
lime_eval.index = lime_eval.index.astype(shap_eval.index.dtype)
eval_idx = shap_eval.index

print("   feature_names: " + str(len(feature_names)))
print("   eval customers: " + str(len(eval_idx)))

X_test_scaled = X_test_scaled[feature_names]

# ------------------------------------------------------------
# C. Per-customer deletion AOPC (Day 4 definition)
# ------------------------------------------------------------
print("")
print(BAR)
print("C. RECOMPUTING DELETION-AOPC PER CUSTOMER")
print(BAR)

baseline = X_test_scaled[feature_names].mean().values
Xe = X_test_scaled.loc[eval_idx, feature_names].values
p0 = xgb_model.predict_proba(X_test_scaled.loc[eval_idx])[:, 1]

def deletion_curve_per_customer(attr_values):
    order = np.argsort(-np.abs(attr_values), axis=1)
    Xm = Xe.copy()
    curve = [p0.copy()]
    rows = np.arange(Xm.shape[0])
    for step in range(FID_K):
        f = order[:, step]
        Xm[rows, f] = baseline[f]
        curve.append(xgb_model.predict_proba(
            pd.DataFrame(Xm, columns=feature_names))[:, 1])
    curve = np.array(curve)
    per_cust = np.abs(curve[0:1] - curve).mean(axis=0)   # mean over steps
    overall = np.abs(curve[0:1] - curve).mean()          # Day 4 scalar
    return per_cust, overall, curve.mean(axis=1)

rng = np.random.default_rng(RNG_SEED)
rand_attr = rng.standard_normal(shap_eval.shape)

print("   SHAP...")
shap_fid, shap_aopc, shap_curve = deletion_curve_per_customer(shap_eval.values)
print("   LIME...")
lime_fid, lime_aopc, lime_curve = deletion_curve_per_customer(lime_eval.values)
print("   Random...")
rand_fid, rand_aopc, rand_curve = deletion_curve_per_customer(rand_attr)

print("")
print("   VERIFICATION against reported Day 4 values")
targets = [('SHAP', shap_aopc, 0.5814210176467896),
           ('LIME', lime_aopc, 0.5377316474914551),
           ('Random', rand_aopc, 0.38310492038726807)]
ok = True
for name, got, want in targets:
    diff = abs(got - want)
    status = "MATCH" if diff < 1e-4 else "MISMATCH"
    if diff >= 1e-4:
        ok = False
    print("   " + name.ljust(8) + " recomputed " + format(got, '.6f')
          + "   reported " + format(want, '.6f') + "   " + status)

print("")
if ok:
    print("   All three reproduce exactly. Per-customer decomposition is valid.")
else:
    print("   WARNING: mismatch. Paste this back before using the CIs.")

# ------------------------------------------------------------
# D. Bootstrap
# ------------------------------------------------------------
print("")
print(BAR)
print("D. BOOTSTRAP FIDELITY (" + str(N_BOOT) + " resamples)")
print(BAR)

n = len(eval_idx)
brng = np.random.default_rng(RNG_SEED)
boot_idx = brng.integers(0, n, size=(N_BOOT, n))

def boot_mean(x):
    d = np.asarray(x)[boot_idx].mean(axis=1)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def boot_paired_diff(x, y):
    d = (np.asarray(x) - np.asarray(y))[boot_idx].mean(axis=1)
    lo = np.percentile(d, 2.5)
    hi = np.percentile(d, 97.5)
    p_two = 2.0 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), lo, hi, max(p_two, 1.0 / N_BOOT)

rows = []
m, lo, hi = boot_mean(shap_fid)
rows.append(['Fidelity - SHAP AOPC', m, lo, hi, ''])
m, lo, hi = boot_mean(lime_fid)
rows.append(['Fidelity - LIME AOPC', m, lo, hi, ''])
m, lo, hi = boot_mean(rand_fid)
rows.append(['Fidelity - Random AOPC', m, lo, hi, ''])

d, lo, hi, pv = boot_paired_diff(shap_fid, lime_fid)
rows.append(['Fidelity - SHAP minus LIME', d, lo, hi, 'p=' + format(pv, '.4f')])
d2, lo2, hi2, pv2 = boot_paired_diff(shap_fid, rand_fid)
rows.append(['Fidelity - SHAP minus Random', d2, lo2, hi2, 'p=' + format(pv2, '.4f')])
d3, lo3, hi3, pv3 = boot_paired_diff(lime_fid, rand_fid)
rows.append(['Fidelity - LIME minus Random', d3, lo3, hi3, 'p=' + format(pv3, '.4f')])

print("")
print("   " + "Quantity".ljust(32) + "Mean".rjust(9) + "CIlow".rjust(11)
      + "CIhigh".rjust(12) + "  sig")
print("   " + "-" * 82)
for label, m, lo, hi, extra in rows:
    print("   " + label.ljust(32) + format(m, '9.4f') + format(lo, '11.4f')
          + format(hi, '12.4f') + "  " + extra)

# ------------------------------------------------------------
# E. Read
# ------------------------------------------------------------
print("")
print(BAR)
print("E. READ")
print(BAR)

d, lo, hi, pv = boot_paired_diff(shap_fid, lime_fid)
if lo > 0:
    print("   SHAP significantly MORE faithful than LIME. diff "
          + format(d, '.4f') + ", 95% CI [" + format(lo, '.4f') + ", "
          + format(hi, '.4f') + "], p=" + format(pv, '.4f'))
elif hi < 0:
    print("   LIME significantly more faithful (unexpected). diff " + format(d, '.4f'))
else:
    print("   Difference NOT significant - CI spans zero. diff "
          + format(d, '.4f') + ", 95% CI [" + format(lo, '.4f') + ", "
          + format(hi, '.4f') + "], p=" + format(pv, '.4f'))
    print("   This must be reported honestly in Chapter 5.")

# ------------------------------------------------------------
# F. Save
# ------------------------------------------------------------
per_fid = pd.DataFrame({
    'customer_id': eval_idx,
    'shap_aopc': shap_fid,
    'lime_aopc': lime_fid,
    'random_aopc': rand_fid,
})
per_fid.to_csv(os.path.join(RESULTS, 'per_customer_fidelity.csv'), index=False)
pd.DataFrame(rows, columns=['quantity', 'mean', 'ci_low', 'ci_high', 'note']).to_csv(
    os.path.join(RESULTS, 'bootstrap_fidelity_summary.csv'), index=False)

print("")
print(BAR)
print("Saved: per_customer_fidelity.csv, bootstrap_fidelity_summary.csv")
print("DONE - paste sections C, D and E back.")
print(BAR)


A. FILES IN MODELS DIRECTORY
         0.5 KB   bootstrap_ci_summary.csv
        25.6 KB   bootstrap_results.pkl
       730.5 KB   churn_segments.csv
         2.5 KB   evaluation_metrics.pkl
       718.3 KB   lime_eval.csv
       264.9 KB   lime_outputs.pkl
         1.9 KB   logistic_regression_tuned.pkl
       655.8 KB   lr_shap_eval.csv
         0.8 KB   metric_summary.csv
         0.4 KB   model_comparison_results.csv
        21.8 KB   per_customer_metrics.csv
    136699.3 KB   random_forest_tuned.pkl
       722.0 KB   rf_shap_eval.csv
         3.8 KB   scaler.pkl
        73.3 KB   segmentation.pkl
         4.0 KB   shap_importance_normalised.csv
     17528.1 KB   shap_outputs.pkl
     48624.1 KB   test_data.pkl
       409.1 KB   xgb_shap_eval.csv
     17416.1 KB   xgb_shap_values.csv
       699.6 KB   xgboost_tuned.pkl

B. LOADING MODEL AND TEST DATA
   loaded test_data.pkl -> <class 'dict'>
   test keys: ['X_test_scaled', 'y_test', 'X_train_resampled', 'y_train_resampled', 'feature

/tmp/ipykernel_834/2224691707.py:37: UserWarning: [17:21:48] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  return pickle.load(fh)


   loaded model: xgboost_tuned.pkl
   feature_names: 66
   eval customers: 500

C. RECOMPUTING DELETION-AOPC PER CUSTOMER
   SHAP...
   LIME...
   Random...

   VERIFICATION against reported Day 4 values
   SHAP     recomputed 0.581421   reported 0.581421   MATCH
   LIME     recomputed 0.537732   reported 0.537732   MATCH
   Random   recomputed 0.383105   reported 0.383105   MATCH

   All three reproduce exactly. Per-customer decomposition is valid.

D. BOOTSTRAP FIDELITY (2000 resamples)

   Quantity                             Mean      CIlow      CIhigh  sig
   ----------------------------------------------------------------------------------
   Fidelity - SHAP AOPC               0.5815     0.5688      0.5938  
   Fidelity - LIME AOPC               0.5378     0.5271      0.5485  
   Fidelity - Random AOPC             0.3832     0.3707      0.3953  
   Fidelity - SHAP minus LIME         0.0437     0.0363      0.0512  p=0.0005
   Fidelity - SHAP minus Random       0.1984     0.1839   